# 02 — Öneri Modeli (Recommender Model)

**Cold-Start Sosyal-Grafik Sızıntısı — Simülasyon Çalışması**

01'in Drive'a kaydettiği veri setlerini okur; üzerinde **standart, sızdırmak için tasarlanmamış** öneri modelleri eğitir; eğitilmiş gömme (embedding) ve ağırlıkları **Drive'a kaydeder.** Sonraki notebook (03 — Saldırgan) bunları okuyup cold→warm taramasını yapacak.

### Döngüselliği (circularity) kıran tasarım — neden bu hileli değil?
İki kanonik mimari kuruyoruz:
- **LightGCN** — yalnız-grafik yapısal gömme (parametresiz komşuluk yayılımı + BPR).
- **FeatureGNN** — yapı **+** düğüm özelliklerini birleştiren GraphSAGE/GraFRank-lite (çok-modlu füzyon).

İkisi de standart **link-prediction** amacıyla eğitilir. Hiçbir yere *"hedefin komşusunu öner"* kuralı koymuyoruz. Sızıntı, ileride şu standart davranıştan **kendiliğinden** doğacak:

> Gözlemci hesabı tanınmadığında (cold-start), sistem makul biçimde **bağlama** düşer — yani "bu profili görüntülediğin için benzer hesaplar" (context-aware / session-based recommendation). Bağlam profili hedef olduğu için, yetkin bir alaka modeli kaçınılmaz olarak hedefin graf-komşularını yukarı taşır.

`link-AUC` ile modelin **gerçekten yetkin bir öneri sistemi** olduğunu (hileli/bozuk değil) gösteriyoruz; warm-start kontrolünün ≈ rastgele çıkması ise sızıntının özellikle cold/bağlam koşuluna özgü olduğunu kanıtlıyor.


## 0. Kurulum ve Drive bağlama

In [ ]:
!pip install -q networkx numpy pandas scipy scikit-learn matplotlib
# torch Colab'da hazır gelir; ayrıca kurmaya gerek yok.

In [ ]:
import os, json, pickle
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
import matplotlib.pyplot as plt

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')   # Colab dışı (yerel test)

DATA_DIR   = os.path.join(ROOT, 'data')
MODELS_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
torch.manual_seed(0); np.random.seed(0)
print("Veri kaynağı :", DATA_DIR)
print("Model çıktısı:", MODELS_DIR)
print("torch", torch.__version__)

## 1. Veri setini Drive'dan oku (01'in şeması, kendi kendine yeterli)

In [ ]:
class SocialGraphDataset:
    def __init__(s, name, G, private, community, node_list, features, meta=None):
        s.name, s.G, s.private, s.community = name, G, private, community
        s.node_list, s.features, s.meta = node_list, features, (meta or {})
    def followers(s, t):  return set(s.G.predecessors(t))
    def following(s, t):  return set(s.G.successors(t))
    def neighborhood(s, t): return s.followers(t) | s.following(t)   # N(t) — GROUND TRUTH

def load_dataset(name, root=DATA_DIR):
    d = os.path.join(root, name)
    G = pickle.load(open(os.path.join(d, "graph.pkl"), "rb"))
    ndf = pd.read_csv(os.path.join(d, "nodes.csv"))
    X = np.load(os.path.join(d, "features.npy"))
    return SocialGraphDataset(name, G, dict(zip(ndf.node_id, ndf["private"])),
                              dict(zip(ndf.node_id, ndf.community)), list(ndf.node_id), X)

## 2. Normalize komşuluk + standart mimariler

In [ ]:
def norm_adj(G, n):
    und = G.to_undirected()
    A = torch.zeros((n, n))
    for u, v in und.edges():
        A[u, v] = 1.0; A[v, u] = 1.0
    A += torch.eye(n)                                  # self-loop
    deg = A.sum(1); dinv = torch.pow(deg, -0.5); dinv[torch.isinf(dinv)] = 0
    Dm = torch.diag(dinv)
    return Dm @ A @ Dm                                  # D^-1/2 (A+I) D^-1/2

class LightGCN(nn.Module):
    # yalnız-grafik: öğrenilebilir taban gömme + parametresiz yayılım
    def __init__(self, n, dim=64, layers=3):
        super().__init__()
        self.emb = nn.Embedding(n, dim); nn.init.normal_(self.emb.weight, std=0.1)
        self.layers = layers
    def forward(self, Ahat):
        e = self.emb.weight; outs = [e]
        for _ in range(self.layers):
            e = Ahat @ e; outs.append(e)
        return torch.stack(outs, 0).mean(0)             # katman birleşimi (mean)

class FeatureGNN(nn.Module):
    # GraphSAGE/GraFRank-lite: kimlik gömme + düğüm özellikleri (çok-modlu füzyon)
    def __init__(self, n, feat_dim, dim=64, layers=2):
        super().__init__()
        self.id_emb = nn.Embedding(n, dim); nn.init.normal_(self.id_emb.weight, std=0.1)
        self.feat_proj = nn.Linear(feat_dim, dim)
        self.W = nn.ModuleList([nn.Linear(2*dim, dim) for _ in range(layers)])
        self.act = nn.ReLU(); self.layers = layers
    def forward(self, Ahat, Xfeat):
        h = self.id_emb.weight + self.feat_proj(Xfeat)
        for l in range(self.layers):
            agg = Ahat @ h
            h = self.act(self.W[l](torch.cat([h, agg], 1)))
        return h

## 3. BPR eğitimi + yetkinlik testi (link-AUC)

**BPR** (Bayesian Personalized Ranking) standart sıralama amacı: gerçek bir bağlantı (pozitif), rastgele bir çiftten (negatif) daha yüksek skor almalı. **link-AUC** ise modelin gerçekten işe yaradığını (komşuları komşu-olmayanlardan ayırabildiğini) gösterir — "hileli değil, yetkin bir öneri sistemi" iddiamızın kanıtı.


In [ ]:
def train_bpr(model, Ahat, pos_pairs, n_nodes, Xfeat=None, epochs=60, bs=1024, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos = torch.tensor(pos_pairs, dtype=torch.long)
    for ep in range(epochs):
        perm = torch.randperm(pos.size(0)); tot = 0.0
        for i in range(0, pos.size(0), bs):
            idx = perm[i:i+bs]; u, p = pos[idx, 0], pos[idx, 1]
            neg = torch.randint(0, n_nodes, (u.size(0),))
            E = model(Ahat) if Xfeat is None else model(Ahat, Xfeat)
            su = (E[u]*E[p]).sum(1); sn = (E[u]*E[neg]).sum(1)
            loss = -torch.log(torch.sigmoid(su - sn) + 1e-9).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        if ep % 20 == 0 or ep == epochs-1:
            print(f"  epoch {ep:3d}  bpr_loss={tot:.3f}")
    with torch.no_grad():
        return (model(Ahat) if Xfeat is None else model(Ahat, Xfeat)).detach()

def link_auc(E, G, n_samples=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    und = G.to_undirected(); edges = list(und.edges()); nodes = list(G.nodes())
    pos = [edges[i] for i in rng.choice(len(edges), size=min(n_samples, len(edges)), replace=False)]
    En = torch.nn.functional.normalize(E, dim=1)
    sc = lambda a, b: float((En[a]*En[b]).sum())
    ps = np.array([sc(u, v) for u, v in pos]); ns = []
    while len(ns) < len(ps):
        a, b = rng.choice(nodes, 2, replace=False)
        if not und.has_edge(a, b): ns.append(sc(a, b))
    ns = np.array(ns)
    return float((ps[:, None] > ns[None, :]).mean())    # P(pos > neg)

## 4. Modelleri eğit ve Drive'a kaydet

In [ ]:
DATASETS = ["sbm_default", "lfr_default"]      # 01'in ürettikleri
EPOCHS = 60
rows = []

for name in DATASETS:
    try:
        ds = load_dataset(name)
    except Exception as e:
        print(f"[atlandı] {name}: {e}"); continue
    n = len(ds.node_list)
    Ahat = norm_adj(ds.G, n)
    Xfeat = torch.tensor(ds.features, dtype=torch.float32)
    pos_pairs = [(u, v) for u, v in ds.G.to_undirected().edges()]

    print(f"\n=== {name} | LightGCN (yalnız-grafik) ===")
    lg = LightGCN(n, dim=64, layers=3)
    E_lg = train_bpr(lg, Ahat, pos_pairs, n, epochs=EPOCHS)
    auc_lg = link_auc(E_lg, ds.G)

    print(f"=== {name} | FeatureGNN (grafik+özellik) ===")
    fg = FeatureGNN(n, ds.features.shape[1], dim=64, layers=2)
    E_fg = train_bpr(fg, Ahat, pos_pairs, n, Xfeat=Xfeat, epochs=EPOCHS)
    auc_fg = link_auc(E_fg, ds.G)

    for mdl_name, E, mdl, auc in [("lightgcn", E_lg, lg, auc_lg),
                                  ("featuregnn", E_fg, fg, auc_fg)]:
        np.save(os.path.join(MODELS_DIR, f"{name}__{mdl_name}__emb.npy"), E.numpy())
        torch.save(mdl.state_dict(), os.path.join(MODELS_DIR, f"{name}__{mdl_name}__state.pt"))
        json.dump({"dataset": name, "model": mdl_name, "dim": 64, "epochs": EPOCHS,
                   "link_auc": round(auc, 4)},
                  open(os.path.join(MODELS_DIR, f"{name}__{mdl_name}__meta.json"), "w"), indent=2)
        rows.append({"dataset": name, "model": mdl_name, "link_auc": round(auc, 3)})

pd.DataFrame(rows)

## 5. Cold→warm kapılı skorlama — *mekanizma*

Bir gözlemci hesap, hedef profili ziyaret edince üretilen "önerilen hesaplar":

$$\text{score}(c \mid \text{gözlemci}, \text{hedef}) = \alpha(h)\cdot \text{sim}(e_{\text{gözlemci}}, e_c) + (1-\alpha(h))\cdot \text{sim}(e_{\text{hedef}}, e_c)$$

- $e_{\text{gözlemci}}$ : gözlemcinin **h** adet bilinen bağlantısının gömmelerinin ortalaması (standart indüktif toplama).
- $\alpha(h) = h/(h+k)$ : gözlemcinin kendi sinyaline duyulan **güven**. $h=0 \Rightarrow \alpha=0 \Rightarrow$ saf **bağlam** (hedef) fallback'i.

> Burada *"hedefin komşularını öner"* gibi bir kural **yok.** Sadece (i) standart güven kapısı ve (ii) bağlam profiline alaka var. Sızıntı bunların kesişiminden **emergent** olarak doğar.


In [ ]:
class Recommender:
    def __init__(self, E, node_list, k_gate=5):
        self.E = torch.nn.functional.normalize(torch.as_tensor(E, dtype=torch.float32), dim=1)
        self.node_list = node_list; self.k_gate = k_gate
    def alpha(self, h): return h / (h + self.k_gate)
    def observer_embedding(self, contacts):
        if len(contacts) == 0: return None
        return torch.nn.functional.normalize(self.E[list(contacts)].mean(0, keepdim=True), dim=1)[0]
    def suggest(self, target, observer_contacts, k=20, exclude=None):
        h = len(observer_contacts); a = self.alpha(h)
        ctx = self.E @ self.E[target]                          # bağlam (hedef) alakası
        if h > 0:
            obs = self.E @ self.observer_embedding(observer_contacts)
            score = a*obs + (1-a)*ctx
        else:
            score = ctx                                        # cold: saf fallback
        score = score.clone(); score[target] = -1e9
        for x in (exclude or []): score[x] = -1e9
        return torch.topk(score, k).indices.tolist()

## 6. Mekanizma demosu — sızıntı *kendiliğinden* mi çıkıyor?

Aynı modelle aynı hedefleri iki gözlemci tipiyle tarıyoruz: **cold** (h=0, geçmişsiz) ve **warm** (hedefle ilgisiz, kendi bağlantıları olan). Cold precision'ın yüksek, warm'un ≈ rastgele çıkması beklenir. *(Tam tarama ve calibrasyon 03'te.)*


In [ ]:
demo = "sbm_default"
ds = load_dataset(demo)
E = np.load(os.path.join(MODELS_DIR, f"{demo}__lightgcn__emb.npy"))
rec = Recommender(E, ds.node_list, k_gate=5)
rng = np.random.default_rng(1)
allnodes = list(ds.G.nodes())
targets = [t for t in ds.node_list if ds.private[t] and ds.G.degree(t) >= 4]

def precision(t, contacts, k=20):
    sugg = rec.suggest(t, contacts, k=k)
    return len(set(sugg) & ds.neighborhood(t)) / k

cold, warm = [], []
for t in targets:
    cold.append(precision(t, set(), k=20))
    wc = set(rng.choice(allnodes, 15, replace=False)) - ds.neighborhood(t) - {t}
    warm.append(precision(t, wc, k=20))

print(f"{demo} | {len(targets)} private hedef | LightGCN")
print(f"  COLD precision@20 = {np.mean(cold):.3f}")
print(f"  WARM precision@20 = {np.mean(warm):.3f}")

plt.figure(figsize=(5,4))
plt.bar(["cold (h=0)", "warm"], [np.mean(cold), np.mean(warm)],
        color=["#C0504D", "#4F81BD"], yerr=[np.std(cold), np.std(warm)], capsize=6)
plt.ylabel("precision@20 (N(t) ile kesişim)")
plt.title("Sızıntı emergent: cold ≫ warm"); plt.tight_layout(); plt.show()

## Sonraki notebook için

**03 — Saldırgan & Metrikler** şu artefaktları Drive'dan okuyacak:
- `models/<veri>__<model>__emb.npy` (gömmeler) — `Recommender` bunlarla kurulur,
- `data/<veri>/...` (ground-truth `N(t)` için).

03'te yapılacaklar: gözlemci geçmiş bütçesi **h**'yi 0→büyük süpürerek **doz-yanıt eğrisi**; tüm private hedefler üzerinde **precision/recall/F1**; rastgele/popülerlik/aynı-topluluk temellerine karşı **saldırgan avantajı**; ve gözlenen ~%72.6 cold-start precision'a **calibrasyon**.
